In [16]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

def transform_dolar_to_pesos(features, features_used):
    for index, value in features["Moneda"].items():
        if value == "U$S":
            features.at[index, "Precio"] = np.float64(features_used.value_dolar * features.at[index, "Precio"])

def transform_pesos_to_dolar(features, features_used):
    for index, value in features["Moneda"].items():
        if value == "$":
            features.at[index, "Precio"] = np.float64(features.at[index, "Precio"] / 844)

def transform_price(features, features_used):
    if features_used.price_type == "Dolar":
        transform_pesos_to_dolar(features, features_used)
    else:
        transform_dolar_to_pesos(features, features_used)

def trasform_years_used(features):
    features["Años de uso"]=features["Año"].apply(lambda x:2024-x)
    features.drop(["Año"],axis=1,inplace=True)
    features[features["Años de uso"] >= 0]

def extract_kilometres(features):
    kilometres = features["Kilómetros"].str.split(" ",expand=True)
    features["Kilómetros"] = pd.to_numeric(kilometres[0],errors="coerce")

def transform_categ_features(features, features_used):
    features_one_hot = features_used
    if len(features_one_hot) != 0: 
        return pd.get_dummies(features,columns=features_one_hot,drop_first=True)
    else:
        return features

def delete_columns(features, features_used):
    columns_delete = features_used
    if len(columns_delete) != 0:
        features.drop(columns_delete,axis=1,inplace=True)

def delete_noise_doors(features):
     features[features["Puertas"] <= 6]

def standar_features(features, features_used):
    features_stadar = features_used

    if len(features_stadar) != 0:
        scaler = StandardScaler()
        features[features_stadar] = scaler.fit_transform(features[features_stadar])

# Opcionales

def transform_version(features):
    frecuencia_modelos = features.groupby(["Marca", "Modelo"]).size().reset_index(name="Frecuencia")

    total_marca = features["Marca"].value_counts().reset_index()
    total_marca.columns = ["Marca", "Total"]

    frecuencia_modelos = frecuencia_modelos.merge(total_marca, on="Marca")

    frecuencia_modelos["Probabilidad"] = frecuencia_modelos["Frecuencia"] / frecuencia_modelos["Total"]

    features.drop(columns=["Marca","Modelo"], inplace=True)

def scalar_color(features, dic_probas):
    for index, value in features["Color"].items():
        features.loc[index, "Color"] = dic_probas[value]

def transform_features_color(features):
    list_color = {"gris":["gris","gray"],"blanco":["blanco","blanca"],"negro":["negro","negra","black"],
                  "plateado":["plateado","plata"],"azul":["azul","blue"],"rojo":["rojo","red"],
                  "marrón":["marrón","café"],"dorado":["dorado"],"verde":["verde"],"celeste":["celeste"],
                  "naranja":["naranja","orange"],"amarillo":["amarillo"],"violeta":["violeta"],
                  "bordó":["bordó"]}
    dic_count = {key: 0 for key in list_color}

    dic_count["otro"] = 0

    for index, value in features["Color"].items():
        if type(value) == str:
            word = value.lower() 
            cond = False
            for key, sub_list_color in list_color.items():
                for color in sub_list_color:
                    if color in word: 
                        features.loc[index, "Color"] = key
                        dic_count[key] += 1
                        cond = True
                        break
                if cond == True:
                    break
            if cond == False:
                features.loc[index, "Color"] = "otro" 
                dic_count["otro"] += 1
        else:
            features.loc[index, "Color"] = "otro"
            dic_count["otro"] += 1


    size_features = features.size

    dic_probas = {key: (value / size_features) for key, value in dic_count.items()}

    scalar_color(features, dic_probas)
    features.drop(columns=["Color"], inplace=True)


def scalar_kilometres(features):
    features["Kilómetros_invertidos"] = -features["Kilómetros"]

    scaler = MinMaxScaler()

    features["Kilómetros_escalados"] = scaler.fit_transform(features[["Kilómetros_invertidos"]])

    features.drop(columns=["Kilómetros_invertidos","Kilómetros"], inplace=True)

def prob_model_for_marca(features):
    conteo_versiones_por_marca = features.groupby("Marca")["Modelo"].count().reset_index()
    conteo_versiones_por_marca.columns = ["Marca", "conteo_versiones"]

    df = features.merge(conteo_versiones_por_marca, on="Marca")

    df["probabilidad"] = 1 / df["conteo_versiones"]

    df = df.drop(columns=["conteo_versiones"])

    return df

def extract_features(path_dataset, features_used):
    features = pd.read_csv(path_dataset)

    transform_price(features, features_used)

    trasform_years_used(features)

    extract_kilometres(features)

    delete_noise_doors(features)

    features = transform_categ_features(features, features_used)

    standar_features(features, features_used)

# Opcionales
    transform_features_color(features)

    transform_version(features)

    scalar_kilometres(features)

    delete_columns(features, features_used)

    print(features)
    return features

In [17]:
features_delete = [
    "Motor",
    "Tipo de carrocería",
    "Título",
    "Con cámara de retroceso",
    "Moneda",
    "Versión"
]

features_one_hot = [
    "Tipo de combustible",
    "Transmisión",
    "Tipo de vendedor",
    "Puertas"
]

In [18]:
import pandas as pd

features = pd.read_csv('pf_suvs_i302_1s2024.csv')

trasform_years_used(features)

extract_kilometres(features)

delete_noise_doors(features)
transform_pesos_to_dolar(features,2)

features = transform_categ_features(features, features_one_hot)

# standar_features(features, features_used)
transform_features_color(features)

transform_version(features)

delete_columns(features, features_delete)

X = features.values
y = features.values


In [20]:
from sklearn.linear_model import Lasso
model = Lasso(alpha=0.1)
model.fit(X, y)
print(model.coef_)

[[ 1.00000000e+00 -0.00000000e+00 -0.00000000e+00  0.00000000e+00
   0.00000000e+00 -0.00000000e+00  0.00000000e+00 -0.00000000e+00
  -0.00000000e+00  0.00000000e+00 -0.00000000e+00  0.00000000e+00
  -0.00000000e+00  0.00000000e+00 -0.00000000e+00  0.00000000e+00
  -0.00000000e+00 -0.00000000e+00 -0.00000000e+00  0.00000000e+00
   0.00000000e+00]
 [-1.02441978e-07  9.99999870e-01 -0.00000000e+00  0.00000000e+00
   0.00000000e+00 -0.00000000e+00  0.00000000e+00 -0.00000000e+00
  -0.00000000e+00  0.00000000e+00 -0.00000000e+00  0.00000000e+00
  -0.00000000e+00  0.00000000e+00 -0.00000000e+00  0.00000000e+00
  -0.00000000e+00 -0.00000000e+00 -0.00000000e+00  0.00000000e+00
   0.00000000e+00]
 [ 6.79749016e-08  8.56372412e-08  9.99999993e-01 -0.00000000e+00
  -0.00000000e+00  0.00000000e+00 -0.00000000e+00  0.00000000e+00
   0.00000000e+00 -0.00000000e+00  0.00000000e+00 -0.00000000e+00
   0.00000000e+00 -0.00000000e+00  0.00000000e+00 -0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.0

In [21]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
model = LinearRegression()
rfe = RFE(model, n_features_to_select=5)  # Elegir el número deseado de características
fit = rfe.fit(X, y)
print("Características seleccionadas:", fit.support_)

Características seleccionadas: [False False False False False  True False  True False False False False
 False False  True False False False  True False  True]


In [22]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
selector = SelectKBest(score_func=chi2, k=5)  # Elegir el número deseado de características
fit = selector.fit(X, y)
print(fit.scores_)

ValueError: Input X must be non-negative.